In [1]:
# Setup
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import torch
from chronos import Chronos2Pipeline

USE_FINETUNED = False

BASE_MODEL = "amazon/chronos-2"
LOCAL_CKPT = "chronos-2-tep-sp-3runs-ctx2048-lora-final"   # saved by TEP-Chronos2-FineTune.ipynb
HF_REPO_ID = "Imker0311/chronos-2-tep-sp-3runs-ctx2048-lora"


def resolve_checkpoint(finetuned):
    if not finetuned:
        return BASE_MODEL
    if Path(LOCAL_CKPT).is_dir():
        return LOCAL_CKPT
    from huggingface_hub import snapshot_download
    return snapshot_download(repo_id=HF_REPO_ID, repo_type="model")


device = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT = resolve_checkpoint(USE_FINETUNED)
kwargs = dict(device_map=device)
if USE_FINETUNED:
    kwargs["import_allowlist"] = ["chronos.chronos2.model"]
pipeline = Chronos2Pipeline.from_pretrained(CHECKPOINT, **kwargs)

MODEL_LABEL = "Chronos-2 fine-tuned" if USE_FINETUNED else "Chronos-2"
MAX_CONTEXT = pipeline.model_context_length
MAX_HORIZON = pipeline.model_prediction_length
print(f"Loaded {MODEL_LABEL} from {CHECKPOINT} on {device}")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loaded Chronos-2 from amazon/chronos-2 on cuda


In [2]:
# Load data
import json

DATA_FILE = "tep_sp_spaced_6min_noisy.h5"


def read_names(ds):
    return [n.decode() if isinstance(n, bytes) else str(n) for n in ds.attrs["names"]]


groups = {}
with h5py.File(DATA_FILE, "r") as f:
    time_h = f["t"][()] / 3600
    for key in ["xmeas", "xmv", "dist", "sp"]:
        groups[key] = (f[key][()], read_names(f[key]))
    if "y" in f:   # true valve positions
        groups["valve"] = (f["y"][:, 38:50], read_names(f["y"])[38:50])
    config = json.loads(f.attrs["config"])
SAMPLE_MIN = config["record_every"] / 60   # sampling interval (min)

nominal = {int(k): v for k, v in config["setpoints"].items()}
CHANGED = sorted({k for k, _, _ in config["setpoint_schedule"]})   # set points changed in this run
t_test = config.get("t_test")                                    # start of the test changes (h)

sp = groups["sp"][0]
off = {k: np.abs(sp[:, k - 1] - nominal[k]) > 1e-9 for k in CHANGED}
sp_change = np.zeros(len(time_h), dtype=int)
for k in reversed(CHANGED):
    sp_change[off[k]] = k

data = {"time_h": time_h, "sp_change": sp_change}
COLUMNS, LABELS = {}, {}
for group, (values, labels) in groups.items():
    COLUMNS[group] = [f"{group}_{i}" for i in range(1, values.shape[1] + 1)]
    for col, v, label in zip(COLUMNS[group], values.T, labels):
        data[col] = v
        LABELS[col] = label

df = pd.DataFrame(data)
df.insert(0, "item_id", "tep")
df.insert(1, "timestamp", pd.date_range("2026-01-01", periods=len(df), freq=f"{config['record_every']}s"))

TARGET_COLUMNS = COLUMNS["xmeas"] + COLUMNS["xmv"]
print(f"{len(df)} samples ({time_h[-1]:.0f} h), {len(TARGET_COLUMNS)} targets, covariate groups: "
      + ", ".join(f"{g} ({len(COLUMNS[g])})" for g in COLUMNS if g not in ["xmeas", "xmv"]))
print(f"changed set points: {CHANGED}, test part from {t_test} h")
df.head()

6798 samples (680 h), 53 targets, covariate groups: dist (12), sp (20), valve (12)
changed set points: [5, 7, 8, 11, 13, 14, 15, 17, 18, 19, 20], test part from 580.6 h


,item_id,timestamp,time_h,sp_change,xmeas_1,xmeas_2,xmeas_3,xmeas_4,xmeas_5,xmeas_6,...,valve_3,valve_4,valve_5,valve_6,valve_7,valve_8,valve_9,valve_10,valve_11,valve_12
0,tep,2026-01-01 00:00:00,0.1,0,0.250570,3646.911201,4529.162678,9.361743,26.955182,42.374646,...,24.683121,61.476138,22.362808,40.095187,38.413928,46.439184,47.490184,40.948926,17.063033,50.0
1,tep,2026-01-01 00:06:00,0.2,0,0.251641,3685.782817,4532.305731,9.392197,27.192827,42.306830,...,24.669825,61.220980,22.282045,39.990620,38.834873,46.076364,47.465945,40.781658,19.495237,50.0
2,tep,2026-01-01 00:12:00,0.3,0,0.224798,3649.568095,4485.146695,9.372377,26.672050,42.430624,...,21.892922,61.453199,22.133312,40.754288,37.087577,47.823150,47.578868,40.150284,16.873578,50.0
3,tep,2026-01-01 00:18:00,0.4,0,0.218406,3639.328339,4472.573385,9.490855,26.897409,42.106486,...,21.619293,61.542878,22.350442,40.748417,34.570768,46.721658,47.682376,40.822785,17.567854,50.0
4,tep,2026-01-01 00:24:00,0.5,0,0.259690,3665.940633,4583.163603,9.290618,27.026282,42.238532,...,25.661680,61.170016,21.863833,42.083198,38.107019,48.403660,47.805239,40.148750,18.059470,50.0


In [3]:
# Test events and forecast windows
events = []   # (start index, end index, set point, new value)
for k in CHANGED:
    on = off[k]
    for i in range(len(on)):
        if on[i] and (i == 0 or not on[i - 1]):
            start = i
        if on[i] and (i == len(on) - 1 or not on[i + 1]):
            events.append((start, i, k, sp[start, k - 1]))
events.sort()

BUFFER = 20   # nominal samples before the event at the start of each forecast

# One forecast per test event, from BUFFER samples before the event up to the start of the next event
rows = []
for n, (s, e, k, v) in enumerate(events):
    if t_test is None or time_h[s] < t_test:
        continue
    split = max(s - BUFFER, 0)
    end = events[n + 1][0] if n + 1 < len(events) else len(df)
    if end <= e:
        raise ValueError(f"event {n} is still running when the next event starts")
    horizon = min(end - split, MAX_HORIZON)
    rows.append({
        "event": n,
        "set point": f"SP {k} {LABELS[f'sp_{k}']}",
        "change (%)": 100 * (v - nominal[k]) / nominal[k],
        "start (h)": time_h[split],
        "horizon (h)": horizon * SAMPLE_MIN / 60,
        "split": split,
        "horizon": horizon,
    })
windows = pd.DataFrame(rows).set_index("event")

EXCLUDED = []   # test events left out of the scores
windows = windows.drop(EXCLUDED)
print(f"{len(windows)} test events")
windows.round(2)

11 test events


,set point,change (%),start (h),horizon (h),split,horizon
event,,,,,,
66,SP 11 Stripper underflow (m3/h),1.74,578.6,10.9,5785,109
67,SP 5 Recycle flow (kscmh),-1.86,587.5,8.9,5874,89
68,SP 14 Reactor feed D (mol%),2.91,594.4,12.3,5943,123
69,SP 20 Product E (mol%),-7.18,604.7,12.8,6046,128
70,SP 18 Reactor temperature (C),0.62,615.5,6.5,6154,65
71,SP 15 Reactor feed E (mol%),-3.99,620.0,12.1,6199,121
72,SP 13 Reactor feed A (mol%),-1.86,630.1,12.8,6300,128
73,SP 17 Reactor level (%),-1.67,640.9,10.0,6408,100
74,SP 19 Purge B (mol%),7.23,648.9,12.8,6488,128


In [4]:
# Covariates and scored variables
USE_DIST = False                 # disturbance signals (12)
USE_SP = True                    # changed set points (11)
USE_VALVE = False                # true valve positions (12)
DROPPED = []
PAST_ONLY = []
COVARIATES_KNOWN_FUTURE = True

ALL_COVARIATES = []
for group, use in [("dist", USE_DIST), ("sp", USE_SP), ("valve", USE_VALVE)]:
    if use and group in COLUMNS:
        ALL_COVARIATES += [f"sp_{k}" for k in CHANGED] if group == "sp" else COLUMNS[group]
ALL_COVARIATES = [c for c in ALL_COVARIATES if c not in DROPPED]


def covariates(window_df):
    """Covariates for one forecast window, leaving out those that are constant over it."""
    used = [c for c in ALL_COVARIATES if window_df[c].nunique() > 1]
    future = [c for c in used if COVARIATES_KNOWN_FUTURE and c not in PAST_ONLY]
    past_only = [c for c in used if c not in future]
    return used, future, past_only


EXCLUDED_VARIABLES = ["xmeas_37"]   # Product D, about 0.017 mol%: near zero, so its percentage error explodes
SCORED = [c for c in COLUMNS["xmeas"] if c not in EXCLUDED_VARIABLES]   # measurements only, no valves
SCALE = df[SCORED].std()    # standard deviation over the whole dataset
print(f"scored variables ({len(SCORED)}): {', '.join(SCORED)}")
print(f"covariates ({len(ALL_COVARIATES)}): {', '.join(ALL_COVARIATES) or '-'}")

scored variables (40): xmeas_1, xmeas_2, xmeas_3, xmeas_4, xmeas_5, xmeas_6, xmeas_7, xmeas_8, xmeas_9, xmeas_10, xmeas_11, xmeas_12, xmeas_13, xmeas_14, xmeas_15, xmeas_16, xmeas_17, xmeas_18, xmeas_19, xmeas_20, xmeas_21, xmeas_22, xmeas_23, xmeas_24, xmeas_25, xmeas_26, xmeas_27, xmeas_28, xmeas_29, xmeas_30, xmeas_31, xmeas_32, xmeas_33, xmeas_34, xmeas_35, xmeas_36, xmeas_38, xmeas_39, xmeas_40, xmeas_41
covariates (11): sp_5, sp_7, sp_8, sp_11, sp_13, sp_14, sp_15, sp_17, sp_18, sp_19, sp_20


In [5]:
# Metrics
QUANTILES = [0.1, 0.5, 0.9]
ALPHA = round(1 - (QUANTILES[-1] - QUANTILES[0]), 6)   # 0.2 for the 10-90% interval
METRICS = ["scaled MAE", "MAPE (%)", "coverage", "scaled Winkler"]


def winkler(actual, lower, upper, alpha):
    """Winkler interval score at every step."""
    width = upper - lower
    below = width + 2 / alpha * (lower - actual)
    above = width + 2 / alpha * (actual - upper)
    return np.where(actual < lower, below, np.where(actual > upper, above, width))


def score_event(forecast, truth_df):
    """Metrics per scored variable for one forecast, forecast[col] = (point forecast, lower, upper)."""
    rows = {}
    for col in SCORED:
        pred, lower, upper = forecast[col]
        actual = truth_df[col].to_numpy()
        rows[col] = {
            "scaled MAE": np.mean(np.abs(pred - actual)) / SCALE[col],
            "MAPE (%)": 100 * np.mean(np.abs(actual - pred) / np.abs(actual)),
            "coverage": np.mean((actual >= lower) & (actual <= upper)),
            "scaled Winkler": np.mean(winkler(actual, lower, upper, ALPHA)) / SCALE[col],
        }
    return pd.DataFrame(rows).T

In [6]:
# Chronos-2 forecast of one window
def run_forecast(context_df, truth_df, used, future, past_only):
    future_df = None
    if future:
        future_df = truth_df[["item_id", "timestamp"] + future]

    pred_df = pipeline.predict_df(
        context_df[["item_id", "timestamp"] + TARGET_COLUMNS + used],
        future_df=future_df,
        prediction_length=len(truth_df),
        quantile_levels=QUANTILES,
        id_column="item_id",
        timestamp_column="timestamp",
        target=TARGET_COLUMNS,
    )
    forecast = {}
    for col in TARGET_COLUMNS:
        sub = pred_df[pred_df["target_name"] == col]
        forecast[col] = (sub["predictions"].to_numpy(), sub[str(QUANTILES[0])].to_numpy(),
                         sub[str(QUANTILES[-1])].to_numpy())
    return forecast

In [7]:
# Context lengths: all available history first, then round steps down
CONTEXT_STEP = 1000
SHORT_CONTEXTS = [500, 250]   # extra steps below the smallest round step

shortest_history = int(min(windows["split"].min(), MAX_CONTEXT))
CONTEXT_LENGTHS = [None] + list(range(shortest_history // CONTEXT_STEP * CONTEXT_STEP, 0, -CONTEXT_STEP)) + SHORT_CONTEXTS   # None is all available
print("context lengths: " + ", ".join("max" if c is None else str(c) for c in CONTEXT_LENGTHS))
print("max is " + ", ".join(f"{min(int(s), MAX_CONTEXT)} samples for event {n}" for n, s in windows["split"].items()))

context lengths: max, 5000, 4000, 3000, 2000, 1000, 500, 250
max is 5785 samples for event 66, 5874 samples for event 67, 5943 samples for event 68, 6046 samples for event 69, 6154 samples for event 70, 6199 samples for event 71, 6300 samples for event 72, 6408 samples for event 73, 6488 samples for event 74, 6596 samples for event 75, 6682 samples for event 76


In [8]:
# Forecast every test event at every context length
rows = []
for context in CONTEXT_LENGTHS:
    label = "max" if context is None else str(context)
    for n, w in windows.iterrows():
        split, horizon = int(w["split"]), int(w["horizon"])
        context_used = min(split, MAX_CONTEXT) if context is None else context
        context_df = df.iloc[split - context_used:split]
        truth_df = df.iloc[split:split + horizon]
        used, future, past_only = covariates(pd.concat([context_df, truth_df]))

        forecast = run_forecast(context_df, truth_df, used, future, past_only)
        scores = score_event(forecast, truth_df).mean()
        rows.append({"context": label, "context used": context_used, "event": n, **scores[METRICS]})
    means = pd.DataFrame(rows[-len(windows):])[METRICS].mean()
    print(f"context {label:>4}: " + ", ".join(f"{m} {means[m]:.3f}" for m in METRICS))
results = pd.DataFrame(rows)

context  max: scaled MAE 0.514, MAPE (%) 2.248, coverage 0.812, scaled Winkler 2.304


context 5000: scaled MAE 0.513, MAPE (%) 2.238, coverage 0.816, scaled Winkler 2.298


context 4000: scaled MAE 0.514, MAPE (%) 2.237, coverage 0.811, scaled Winkler 2.301


context 3000: scaled MAE 0.529, MAPE (%) 2.313, coverage 0.806, scaled Winkler 2.398


context 2000: scaled MAE 0.536, MAPE (%) 2.352, coverage 0.808, scaled Winkler 2.436


context 1000: scaled MAE 0.568, MAPE (%) 2.540, coverage 0.777, scaled Winkler 2.673


context  500: scaled MAE 0.602, MAPE (%) 2.696, coverage 0.749, scaled Winkler 2.871


context  250: scaled MAE 0.631, MAPE (%) 2.859, coverage 0.735, scaled Winkler 3.108


In [9]:
# Scores per context length and event, averaged over the scored variables
print(f"{MODEL_LABEL}: mean over {len(SCORED)} measurements per event")
results.set_index(["context", "event"]).round(3)

Chronos-2: mean over 40 measurements per event


context used  scaled MAE  MAPE (%)  coverage  scaled Winkler
context event                                                              
max     66             5785       0.555     2.378     0.808           2.350
        67             5874       0.469     1.928     0.850           2.170
        68             5943       0.445     1.866     0.871           2.055
        69             6046       0.430     1.823     0.885           2.001
        70             6154       0.507     2.013     0.799           2.249
...                     ...         ...       ...       ...             ...
250     72              250       0.479     2.225     0.808           2.178
        73              250       0.685     3.075     0.570           3.621
        74              250       0.696     3.420     0.733           3.504
        75              250       0.639     2.820     0.715           2.966
        76              250       0.777     3.779     0.602           4.093

[88 rows x 5 columns]

In [10]:
# Average over all test events per context length
summary = results.groupby("context", sort=False)[["context used"] + METRICS].mean()
print(f"{MODEL_LABEL}: mean over {len(windows)} test events ({', '.join(map(str, windows.index))})")
summary.round(3)

Chronos-2: mean over 11 test events (66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76)


,context used,scaled MAE,MAPE (%),coverage,scaled Winkler
context,,,,,
max,6225.0,0.514,2.248,0.812,2.304
5000,5000.0,0.513,2.238,0.816,2.298
4000,4000.0,0.514,2.237,0.811,2.301
3000,3000.0,0.529,2.313,0.806,2.398
2000,2000.0,0.536,2.352,0.808,2.436
1000,1000.0,0.568,2.540,0.777,2.673
500,500.0,0.602,2.696,0.749,2.871
250,250.0,0.631,2.859,0.735,3.108


In [11]:
# Save scores
RESULTS_FILE = f"Results/Context_setpointforecast_chronos2_{os.path.splitext(DATA_FILE)[0]}.h5"

os.makedirs("Results", exist_ok=True)
contexts = list(summary.index)
with h5py.File(RESULTS_FILE, "w") as f:
    f.attrs["model"] = MODEL_LABEL
    f.attrs["settings"] = json.dumps({
        "data_file": DATA_FILE,
        "buffer": BUFFER,
        "quantiles": QUANTILES,
        "scored": SCORED,
        "covariates": ALL_COVARIATES,
        "past_only": PAST_ONLY,
        "excluded_events": EXCLUDED,
        "context_step": CONTEXT_STEP,
        "short_contexts": SHORT_CONTEXTS,
    })
    f.create_dataset("context", data=contexts)
    f.create_dataset("context_used", data=summary["context used"].to_numpy())   # mean over events
    f.create_dataset("event", data=windows.index.to_numpy())
    f.create_dataset("split", data=windows["split"].to_numpy())
    f.create_dataset("horizon", data=windows["horizon"].to_numpy())
    for name in METRICS:
        table = results.pivot(index="context", columns="event", values=name).loc[contexts, windows.index]
        f.create_dataset(f"per_event/{name}", data=table.to_numpy())        # (context, event)
        f.create_dataset(f"average/{name}", data=summary[name].to_numpy())  # (context,)
print(f"Saved {MODEL_LABEL} scores to {RESULTS_FILE}")

Saved Chronos-2 scores to Results/Context_setpointforecast_chronos2_tep_sp_spaced_6min_noisy.h5
